<center><h1>A notebook for training the models for classification</h1></center>

# Imports

All necessary imports for performing classificator training

In [1]:
import numpy as np

import torch
from torch import nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm.auto import tqdm

import helper

# Dataset

The definition of transforms applied to the images

In [2]:
TF = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((300, 300)),
    transforms.Normalize((0.5,), (0.5,))
])

Loading the dataset from directory using the self-written module helper.py 

In [3]:
dataset = helper.load_dataset(path='datasets/all13-r-45.pth', transform=TF)

# Constants 

The definitions of constants used for training

In [4]:
BATCH_SIZE = 16
EPOCHS = 100

HIDDEN_LAYER_FEATURES = [32, 64, 96]

LEARNING_RATE = 0.001

CLASSES = np.max(np.unique(dataset.labels)) + 1

EPS = 1e-6

MODEL_NAME = "all13-r-45-100e"

Initializing the GPU for training

In [5]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

Changing the size of displayed figures

In [6]:
plt.rcParams['figure.figsize'] = [12, 8]

# Functions 
A special function to plot the loss curve

In [7]:
def plot_loss_curve(loss_curve, epochs=EPOCHS, save=False, name=None):
    plt.plot(range(1, epochs + 1), list(loss_curve), label="loss curve", color="red")

    plt.ylabel("value")
    plt.xlabel("epoch")
    plt.xticks(range(0, epochs + 1, 10))

    plt.legend()
    plt.grid()
    
    if save:
        plt.savefig(f"figures/{name}_loss.png")
        
    plt.show()

A special function to plot accuracy curves 

In [8]:
def plot_accuracy_curves(train_curve, test_curve, epochs=EPOCHS, save=False, name=None):
    plt.plot(range(1, epochs + 1), list(train_curve), label="train curve accuracy", color="red")
    plt.plot(range(1, epochs + 1), list(test_curve), label="test curve accuracy", color="black")

    plt.ylabel("accuracy")
    plt.xlabel("epoch")
    plt.xticks(range(0, epochs + 1, 10))

    plt.legend()
    plt.grid()
    
    if save:
        plt.savefig(f"figures/{name}_acc.png")
         
    plt.show()

A function used for evaluating the model on the specified loader
It computes following metrics: accuracy, precision, recall, and f1-score

In [9]:
def evaluate(model, loader, device):
    metrics = defaultdict(lambda: list())
    correct = 0
    total = 0

    for i, (X, y)  in tqdm(enumerate(loader), total=len(loader), leave=False):
        X, y = X.to(device), y.to(device)

        logits = model(X)
        predictions = (torch.sigmoid(logits) > 0.5).sum(dim=1)
        
        correct += (predictions == y).sum().item()
        total += y.size(0)

        y_true_np = y.cpu().data.numpy()
        y_pred_np = predictions.cpu().data.numpy()

        metrics['precision'].append(precision_score(y_true_np, y_pred_np, average='micro'))
        metrics['recall'].append(recall_score(y_true_np, y_pred_np, average='micro'))
        metrics['f1_score'].append(f1_score(y_true_np, y_pred_np, average='micro'))

    metrics['accuracy'] = correct / total

    metrics['precision'] = np.mean(metrics['precision'])
    metrics['recall'] = np.mean(metrics['recall'])
    metrics['f1_score'] = np.mean(metrics['f1_score'])

    return metrics

# Training

## Train-Test Split
It utilizes the special function to perform split implemented in helper module.

In [10]:
train_dataset, test_dataset = helper.split(dataset, test_size=0.2)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## A dictionary for saving history

In [11]:
history = defaultdict(lambda:list())

## Architecture

The architecture of model defined as a separate class

In [12]:
class CNN(nn.Module):
    def __init__(self, output_shape: int):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, HIDDEN_LAYER_FEATURES[0], 3, padding=1)
        self.conv2 = nn.Conv2d(HIDDEN_LAYER_FEATURES[0], HIDDEN_LAYER_FEATURES[1], 3, padding=1)
        self.conv3 = nn.Conv2d(HIDDEN_LAYER_FEATURES[1], HIDDEN_LAYER_FEATURES[2], 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.aa_pool = nn.AdaptiveAvgPool2d((3, 3))
        self.dropout = nn.Dropout(0.1)
        self.fc1 = nn.Linear(HIDDEN_LAYER_FEATURES[2] * 3 * 3 , 256)
        self.fc2 = nn.Linear(256, output_shape) 

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.aa_pool(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(self.dropout(x)))
        x = self.fc2(x)
        return x

## Loss function

Loss function is implemented as a separate class, either.
It is specially designed to take the ordinal relationship into account

In [13]:
class LinkLoss(nn.Module):
    def __init__(self, num_classes=CLASSES):
        super(LinkLoss, self).__init__()
        self.num_classes = num_classes

    def forward(self, logits, targets):
        batch_size = targets.size(0)
        
        targets = targets.unsqueeze(1).expand(batch_size, self.num_classes)
        
        mask = torch.arange(self.num_classes).unsqueeze(0).expand(batch_size, self.num_classes).to(targets.device)
        mask = mask < targets

        logits = torch.sigmoid(logits)

        loss = -(mask.float() * (torch.log(logits + EPS)) + (1 - mask.float()) * (torch.log(1 - logits + EPS))).sum(dim=1)
        
        return loss.mean()

criterion = LinkLoss()

## Initialization of the model

Here, we initialize the model and move it to the GPU
Moreover, the optimizer with the specified learning rate is defined here either

In [14]:
model = CNN(output_shape=CLASSES).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

A model has 320834 learning parameters

In [15]:
sum(p.numel() for p in model.parameters())

320834

## Training

A special loop for training 
For every epoch we compute metrics for testing and training set

In [16]:
for epoch in range(EPOCHS):
    print(f"epoch {epoch + 1} ({epoch + 1}/{EPOCHS}):")

    for i, data in tqdm(enumerate(train_loader), total=len(train_loader), leave=False):
        inputs, labels = data
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_metrics = evaluate(model, train_loader, DEVICE)
    test_metrics = evaluate(model, test_loader, DEVICE)

    history['train_accuracy'].append(train_metrics['accuracy'])
    history['test_accuracy'].append(test_metrics['accuracy'])

    history['train_precision'].append(train_metrics['precision'])
    history['test_precision'].append(test_metrics['precision'])

    history['train_recall'].append(train_metrics['recall'])
    history['test_recall'].append(test_metrics['recall'])

    history['train_f1_score'].append(train_metrics['f1_score'])
    history['test_f1_score'].append(test_metrics['f1_score'])

    history['loss'].append(loss.cpu().data.numpy())

    print("train accuracy:", history['train_accuracy'][-1])
    print("test accuracy:", history['test_accuracy'][-1])

    print("loss:", history['loss'][-1], end="\n\n")


epoch 1 (1/100):


  0%|          | 0/866 [00:00<?, ?it/s]

  0%|          | 0/866 [00:00<?, ?it/s]

  0%|          | 0/217 [00:00<?, ?it/s]

train accuracy: 0.022105035035758145
test accuracy: 0.022247905229702398
loss: 36.923943

epoch 2 (2/100):


  0%|          | 0/866 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Results

Graphs to evaluate model's performance

In [17]:
# plot_accuracy_curves(history["train_accuracy"], history["test_accuracy"], save=True, name=f"{MODEL_NAME}")

In [18]:
# plot_loss_curve(history["loss"], save=True, name=f"{MODEL_NAME}")

Eventually, the trained model is saved to file

In [19]:
# torch.save(model.state_dict(), f'models/{MODEL_NAME}_state_dict.pth')